# Information Retrieval Systems
## Phase 1

---
> Eleni Kechrioti p3210078@aueb.gr <br /> 
> Dejvid Isufaj p3210056@aueb.gr

### Introduction

This project focuses on the implementation and evaluation of an information retrieval system based on a text indexing approach. The goal is to efficiently search for relevant documents in response to specific queries, and to measure the system’s performance using the Mean Average Precision (MAP) metric.

For evaluation, the `trec_eval` tool was utilized, which allows comparison of the system’s retrieved results against a ground truth set of relevance judgments. This evaluation is essential for assessing and improving the accuracy and relevance of the search results provided to the user.

If you don't already have those installed, you should do it as we are going to use them below.

In [1]:
!pip install nltk elasticsearch


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup and Imports

The following Python code sets up the necessary imports and downloads required NLTK resources for text preprocessing, as well as imports Elasticsearch libraries for indexing and bulk operations.

In [2]:
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
import json
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk


# Download required NLTK datasets
nltk.download('stopwords')    # Common stopwords for filtering
nltk.download('punkt')        # Tokenizer models for splitting text
nltk.download('punkt_tab')    # Additional tokenizer data (tabs handling)
nltk.download('wordnet')      # WordNet lexical database for lemmatization
nltk.download('omw-1.4')      # Open Multilingual WordNet (needed for some lemmatization)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\giwrg\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\giwrg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\giwrg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\giwrg\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\giwrg\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## Text Preprocessing Function

The `preprocess_text` function normalizes input text by:

- Converting to lowercase  
- Removing punctuation  
- Removing English stopwords  
- Tokenizing the text  
- Applying both lemmatization and stemming  

It returns a dictionary containing the tokenized text and lists of lemmatized and stemmed tokens.


In [7]:
def preprocess_text(text):

  text = text.lower()
  text = text.translate(str.maketrans("", "", string.punctuation))

  stop_words = set(stopwords.words("english"))
  words = text.split()
  filtered_words = [word for word in words if word not in stop_words]
  text = " ".join(filtered_words)
  stemmer = PorterStemmer()
  lemmatizer = WordNetLemmatizer()

  word_tokens = word_tokenize(text)
  word_lemma = [lemmatizer.lemmatize(word) for word in word_tokens]
  word_stem =  [stemmer.stem(word) for word in word_tokens]

  return {'text':word_tokens, 'word_lemma': word_lemma, 'word_stem': word_stem}

Instantiate an Elasticsearch client connected to the local server at port 9200. You should first run the elasticsearch.bat

In [8]:
client = Elasticsearch("http://localhost:9200")

We define a custom Elasticsearch index mapping with advanced settings:

- **Similarity:** We use the BM25 similarity algorithm given by ElasticSearch. While TF-IDF is a simple and intuitive method for weighting terms based on their frequency and rarity, BM25 offers an advanced and fine-tuned method that considers additional factors such as document length and frequency saturation.

- **Analysis:** We use the built-in English analyzer for both indexing (`default`) and searching (`default_search`). Also we removed `case specific stopwords` and stemmed the tokens using ElasticSearch's `porter stemmer`.

- **Mappings:** We specify fields like `title`, `text`, `authors`, `year`, `references`, and `cited_by` as text fields. The fields `title` and `text`, each with `analyzer: scidocs_analyzer`, `similarity: BM25`, and both copy their content to `allContent` to implement full text search. The field `allContent` also uses `scidocs_analyzer + BM25`, giving a unified field for “search across everything.”   

Finally, we create an Elasticsearch index called `corpus_index` with these customized settings and mappings.

In [29]:
custom_mapping = {
  "settings": {
        "analysis": {
        "filter": {
            "scidocs_stop": {
            "type": "stop",
            "stopwords": [
                "the", "and", "of", "in", "to", "a", "is", "for", "on", "that",
                "with", "as", "by", "an", "at", "be", "this", "from", "or", "are",
                "was", "which", "it", "we", "study", "results", "method", "methods",
                "figure", "table", "data", "analysis", "based", "using", "show",
                "shown", "et", "al", "also", "these", "those", "their", "were",
                "may", "can", "used", "use", "such", "have", "has", "had"
            ]
            },
            "english_stemmer": {
            "type": "porter_stem",
            "language": "english"
            }
        },
        "analyzer": {
            "scidocs_analyzer": {
            "type": "custom",
            "tokenizer": "standard",
            "filter": [
                "lowercase",
                "scidocs_stop",
                "english_stemmer"
            ]
            }
        }
        }
    },
  "mappings": {
        "properties": {
            "title": {
                "type": "text",
                "similarity": "BM25",
                "analyzer": "scidocs_analyzer",
                "copy_to": "allContent"
            },
            "text": {
                "type": "text",
                "similarity": "BM25",
                "analyzer": "scidocs_analyzer",
                "copy_to": "allContent"
            },
            "authors": {
                "type": "text",
            },
            "year": {
                "type": "text",
            },
            "references": {
                "type": "text",
            },
            "cited_by": {
                "type": "text",
            },
            "allContent": {
                "type": "text",
                "analyzer": "scidocs_analyzer",
                "similarity": "BM25"
            }
        }
    }
}


client.indices.create(index='corpus_index', body=custom_mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'corpus_index'})

## Document Loading, Preprocessing, and Bulk Indexing into Elasticsearc

We load all JSON documents from the `corpus.jsonl` file line by line into a list.

For each document, we extract metadata fields such as authors, year, citations (`cited_by`), and references.

We preprocess the document's text by applying tokenization, lemmatization, and stemming, but we the tokens as the main text field and let ElasticSearch do th stemming.

We construct a dictionary for each document with the required Elasticsearch indexing format, including the index name, document ID, and source fields.

Finally, we use Elasticsearch’s bulk API to efficiently index all documents into the `corpus_index` and confirm the total number of indexed documents.

In [28]:
# # delete the index anytime
# client.indices.delete(index="corpus_index")

ObjectApiResponse({'acknowledged': True})

In [30]:

all_json_data = [] 
with open("scidocs/corpus.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        json_data=json.loads(line)
        all_json_data.append(json_data)


documents = []
for doc in all_json_data:
        authors = doc['metadata']['authors']
        year = doc['metadata']['year']
        cited_by = doc['metadata']['cited_by']
        references = doc['metadata']['references']

        processed_text = preprocess_text(doc['text'])
        lemmas = processed_text['word_lemma']
        stems = processed_text['word_stem']
        text = processed_text['text']
        document = {
            '_index': 'corpus_index',
            '_id':    doc["_id"],
            '_source':{
                'title': doc['title'],
                'text': text,
                'authors': authors,
                'year': year,
                'references': references,
                'cited_by': cited_by
            }
        }
        documents.append(document)

bulk(client, documents, refresh=True)
print(f"Indexed {len(all_json_data)} documents in corpus_index.")


Indexed 25657 documents in corpus_index.


## Search Results Formatting and Display

We define a function to neatly format and display the search results returned by Elasticsearch. If no results are found, we inform the user accordingly. Otherwise, for each document in the search hits, we extract relevant fields such as the document ID, relevance score, text content, title, authors, publication year, references, and citations. These details are then presented in a clear, readable format to facilitate quick understanding of each search result.

In [31]:
def pretty_search_response(response):
    if len(response["hits"]["hits"]) == 0:
        print("Your search returned no results.")
    else:
        for hit in response["hits"]["hits"]:
            id = hit["_id"]
            score = hit["_score"]
            text = hit["_source"]["text"]
            title = hit["_source"]["title"]
            authors = hit["_source"]["authors"]
            year = hit["_source"]["year"]
            references = hit["_source"]["references"]
            cited_by = hit["_source"]["cited_by"]
            
            pretty_output = f"\nID: {id}\nScore: {score}\nText: {text}\nTitle: {title}\nAuthors: {authors}\nYear: {year}\nReferences: {references}\nCited_by: {cited_by}\n"

            print(pretty_output)

In [32]:
search_results = client.search(
    index="corpus_index",
    size=20,
    query = {"match": {"allContent": "recurrent"}},
)

pretty_search_response(search_results)


ID: 632589828c8b9fca2c3a59e97451fde8fa7d188d
Score: 7.6826944
Text: ['evolutionary', 'recurrent', 'network', 'automates', 'design', 'recurrent', 'neuralfuzzy', 'networks', 'using', 'new', 'evolutionary', 'learning', 'algorithm', 'proposed', 'paper', 'new', 'evolutionary', 'learning', 'algorithm', 'based', 'hybrid', 'genetic', 'algorithm', 'ga', 'particle', 'swarm', 'optimization', 'pso', 'thus', 'called', 'hgapso', 'hgapso', 'individuals', 'new', 'generation', 'created', 'crossover', 'mutation', 'operation', 'ga', 'also', 'pso', 'concept', 'elite', 'strategy', 'adopted', 'hgapso', 'upperhalf', 'bestperforming', 'individuals', 'population', 'regarded', 'elites', 'however', 'instead', 'reproduced', 'directly', 'next', 'generation', 'elites', 'first', 'enhanced', 'group', 'constituted', 'elites', 'regarded', 'swarm', 'elite', 'corresponds', 'particle', 'within', 'regard', 'elites', 'enhanced', 'pso', 'operation', 'mimics', 'maturing', 'phenomenon', 'nature', 'enhanced', 'elites', 'consti

### Configuration Variables

We set the key configuration variables for our evaluation process:

- **queries_file**: Path to the JSONL file containing the queries to be run against the index.
- **run_file**: Path where the search results will be saved in TREC run file format.
- **index_name**: Name of the Elasticsearch index we will query.
- **top_k**: Number of top results to retrieve for each query during search.


In [33]:
queries_file = "scidocs/queries.jsonl"
run_file = "trec_eval/my_results"
index_name = "corpus_index"
top_k = [20, 30, 50]

### Running Queries and Generating TREC Run File

We iterate over each query from the JSONL file and perform a search on the Elasticsearch index. For each query:

- Extract the query ID and query text.
- Perform a search on the specified index, retrieving the top *k* results.
- For each retrieved document, write an entry to the run file in TREC format, including:
  - Query ID
  - A constant "Q0" (required by TREC format)
  - Document ID
  - Rank of the document in the results
  - Relevance score (rounded to 4 decimals)
  - Run tag (to identify this experiment)

This run file will be used later for evaluation with `trec_eval`.

In [34]:

run_tag = "my_run"

for topk in top_k:
    file_path = run_file + str(topk) + ".run"
    with open(queries_file, "r", encoding="utf-8") as qf, open(file_path, "w", encoding="utf-8") as rf:
        for line in qf:
            query_obj = json.loads(line)
            query_id = query_obj["_id"]
            query_text = query_obj["text"]
    
            
            response = client.search(
                index=index_name,
                size=topk,
                query={
                    "bool": {
                        "should": [
                            {"match": { "title": {"query": query_text,"boost": 1}}},
                            {"match": {"text": {"query": query_text,"boost": 1}}},
                            {"match": {"allContent": {"query": query_text,"boost": 1}}}
                        ]
                    }     
                }
            )
            for rank, hit in enumerate(response["hits"]["hits"], start=1):
                doc_id = hit["_id"]
                score = hit["_score"]
                rf.write(f"{query_id} Q0 {doc_id} {rank} {score:.4f} {run_tag}\n")
    



### Converting Qrels File to TREC Format

We read the original qrels file (`test.tsv`), which contains relevance judgments in a tab-separated format with a header. For each line after the header, we:

- Parse the query ID, document ID, and relevance score.
- Write these fields to a new file in TREC evaluation format, where the second column is always `0` (a placeholder required by TREC).

This reformatted qrels file will be compatible with the `trec_eval` tool for evaluation.

In [35]:
with open("scidocs/qrels/test.tsv") as f_in, open("trec_eval/test_trec.tsv", "w") as f_out:
    next(f_in)
    for line in f_in:
        parts = line.strip().split("\t")
        query_id, doc_id, score = parts
        f_out.write(f"{query_id} 0 {doc_id} {score}\n")


We run the `trec_eval` tool to evaluate the retrieval performance by computing the mean average precision (MAP) using our relevance judgments (`test_trec.tsv`) and the search results (`my_results.run`).

In [36]:
files = ['my_results20.run','my_results30.run','my_results50.run']
for f in files:
    print(f'Trec Eval Evaluation for file {f}')
    !.\trec_eval\trec_eval -m map trec_eval/test_trec.tsv trec_eval/{f} 2>null
    !.\trec_eval\trec_eval -m P.5 trec_eval/test_trec.tsv trec_eval/{f} 2>null
    !.\trec_eval\trec_eval -m P.5 trec_eval/test_trec.tsv trec_eval/{f} 2>null
    !.\trec_eval\trec_eval -m P.10 trec_eval/test_trec.tsv trec_eval/{f} 2>null
    !.\trec_eval\trec_eval -m P.15 trec_eval/test_trec.tsv trec_eval/{f} 2>null
    !.\trec_eval\trec_eval -m P.20 trec_eval/test_trec.tsv trec_eval/{f} 2>null

Trec Eval Evaluation for file my_results20.run
map                   	all	0.1019
P_5                   	all	0.1160
P_5                   	all	0.1160
P_10                  	all	0.0826
P_15                  	all	0.0649
P_20                  	all	0.0541
Trec Eval Evaluation for file my_results30.run
map                   	all	0.1051
P_5                   	all	0.1160
P_5                   	all	0.1160
P_10                  	all	0.0826
P_15                  	all	0.0649
P_20                  	all	0.0541
Trec Eval Evaluation for file my_results50.run
map                   	all	0.1078
P_5                   	all	0.1160
P_5                   	all	0.1160
P_10                  	all	0.0826
P_15                  	all	0.0649
P_20                  	all	0.0541
